# Лабораторная работа №2: Предобработка данных
## Набор данных: Titanic (OpenML)

### 📌 Описание датасета

**Датасет Titanic** содержит информацию о пассажирах легендарного лайнера. Целевая переменная — `survived` (выжил или нет).

**Признаки:**
- `pclass` — класс билета (1, 2, 3) — **категориальный**
- `sex` — пол (male/female) — **категориальный**
- `age` — возраст (float) — **числовой, есть пропуски**
- `sibsp` — кол-во siblings/spouses aboard — числовой
- `parch` — кол-во parents/children aboard — числовой
- `fare` — стоимость билета (float) — **числовой, есть пропуски**
- `embarked` — порт посадки (C, Q, S) — **категориальный, есть пропуски**

**Почему этот датасет:**
- ✅ Есть пропуски (Age, Fare, Embarked)
- ✅ Есть категориальные признаки (Sex, Embarked, Pclass)
- ✅ Числовые признаки в разных масштабах (Age, Fare)
- ✅ Компактный (~1300 строк)
- ✅ Классический, много готовых примеров

### Задачи лабораторной:
1. **Обработка пропусков** — анализ, удаление, заполнение (mean, median, mode, constant)
2. **Кодирование категориальных признаков** — Label Encoding, One-Hot Encoding
3. **Масштабирование данных** — StandardScaler, MinMaxScaler, RobustScaler

In [ ]:
# ЯЧЕЙКА 1: ИМПОРТ БИБЛИОТЕК

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    StandardScaler,
    MinMaxScaler,
    RobustScaler
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Настройка отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Настройка графиков
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("✅ Все библиотеки загружены!")

In [ ]:
# ЯЧЕЙКА 2: ЗАГРУЗКА ДАННЫХ

# Загружаем датасет Titanic с OpenML
X, y = fetch_openml("titanic", version=1, as_frame=True, return_X_y=True)

# Объединяем признаки и целевую переменную для удобства
df = X.copy()
df['survived'] = y

print(f"📊 Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")
print("\n📋 Первые 5 строк:")
display(df.head())

print("\n📋 Информация о типах данных:")
df.info()

In [ ]:
# ЯЧЕЙКА 3: АНАЛИЗ ПРОПУСКОВ

print("=" * 60)
print("📊 АНАЛИЗ ПРОПУСКОВ В ДАННЫХ")
print("=" * 60)

# Количество пропусков по каждому столбцу
missing_counts = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Пропуски': missing_counts,
    'Доля, %': missing_pct
})
missing_df = missing_df[missing_df['Пропуски'] > 0].sort_values('Пропуски', ascending=False)

print("\n📌 Столбцы с пропусками:")
display(missing_df)

# Визуализация пропусков
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(missing_df.index, missing_df['Пропуски'], color='#FF6B6B', edgecolor='black')
ax.set_xlabel('Количество пропусков')
ax.set_title('Распределение пропусков по столбцам')
for i, (idx, row) in enumerate(missing_df.iterrows()):
    ax.text(row['Пропуски'] + 2, i, f"{row['Пропуски']} ({row['Доля, %']}%)", va='center')
plt.tight_layout()
plt.show()

### 📌 Часть 1: Обработка пропусков

In [ ]:
# ЯЧЕЙКА 4: ВАРИАНТ 1 — УДАЛЕНИЕ СТРОК С ПРОПУСКАМИ

print("=" * 60)
print("🔹 ВАРИАНТ 1: УДАЛЕНИЕ СТРОК С ПРОПУСКАМИ (listwise deletion)")
print("=" * 60)

df_dropped_rows = df.copy()
initial_rows = len(df_dropped_rows)

# Удаляем все строки, содержащие хотя бы один пропуск
df_dropped_rows = df_dropped_rows.dropna()

print(f"📌 Было строк: {initial_rows}")
print(f"📌 Стало строк: {len(df_dropped_rows)}")
print(f"📌 Удалено строк: {initial_rows - len(df_dropped_rows)} ({((initial_rows - len(df_dropped_rows)) / initial_rows * 100):.1f}%)")

print("\n⚠️ Недостаток: теряется слишком много данных — не лучший вариант.")

In [ ]:
# ЯЧЕЙКА 5: ВАРИАНТ 2 — ЗАПОЛНЕНИЕ СРЕДНИМ (MEAN IMPUTATION)

print("=" * 60)
print("🔹 ВАРИАНТ 2: ЗАПОЛНЕНИЕ СРЕДНИМ ЗНАЧЕНИЕМ (Mean Imputation)")
print("=" * 60)

# Выбираем числовые столбцы с пропусками
numeric_cols_with_na = ['age', 'fare']

df_mean_imp = df.copy()

for col in numeric_cols_with_na:
    mean_val = df_mean_imp[col].mean()
    df_mean_imp[col] = df_mean_imp[col].fillna(mean_val)
    print(f"📌 {col}: пропусков заполнено {df[col].isnull().sum()}, среднее = {mean_val:.2f}")

print("\n✅ Пропуски в числовых признаках заполнены средними.")
print("⚠️ Недостаток: среднее чувствительно к выбросам.")

In [ ]:
# ЯЧЕЙКА 6: ВАРИАНТ 3 — ЗАПОЛНЕНИЕ МЕДИАНОЙ (MEDIAN IMPUTATION)

print("=" * 60)
print("🔹 ВАРИАНТ 3: ЗАПОЛНЕНИЕ МЕДИАНОЙ (Median Imputation)")
print("=" * 60)

df_median_imp = df.copy()

for col in numeric_cols_with_na:
    median_val = df_median_imp[col].median()
    df_median_imp[col] = df_median_imp[col].fillna(median_val)
    print(f"📌 {col}: медиана = {median_val:.2f}")

print("\n✅ Пропуски в числовых признаках заполнены медианой.")
print("✅ Медиана устойчива к выбросам — лучше для данных с аномалиями.")

In [ ]:
# ЯЧЕЙКА 7: ВАРИАНТ 4 — ЗАПОЛНЕНИЕ МОДОЙ (MODE IMPUTATION) ДЛЯ КАТЕГОРИАЛЬНЫХ

print("=" * 60)
print("🔹 ВАРИАНТ 4: ЗАПОЛНЕНИЕ МОДОЙ (Mode Imputation) для категориальных")
print("=" * 60)

df_mode_imp = df.copy()

# Для категориального признака embarked
mode_val = df_mode_imp['embarked'].mode()[0]
df_mode_imp['embarked'] = df_mode_imp['embarked'].fillna(mode_val)
print(f"📌 embarked: мода = '{mode_val}', заполнено {df['embarked'].isnull().sum()} пропусков")

print("\n✅ Категориальные пропуски заполнены модой (самым частым значением).")
print("✅ Логично: если порт неизвестен, берём самый популярный.")

In [ ]:
# ЯЧЕЙКА 8: ВАРИАНТ 5 — ЗАПОЛНЕНИЕ КОНСТАНТОЙ (Constant Imputation)

print("=" * 60)
print("🔹 ВАРИАНТ 5: ЗАПОЛНЕНИЕ КОНСТАНТОЙ (Constant Imputation)")
print("=" * 60)

df_const_imp = df.copy()

# Для числовых — заполняем -1 (сигнал о пропуске)
for col in numeric_cols_with_na:
    df_const_imp[col] = df_const_imp[col].fillna(-1)
    print(f"📌 {col}: заполнено {-1}")

# Для категориальных — заполняем 'unknown'
df_const_imp['embarked'] = df_const_imp['embarked'].fillna('unknown')
print(f"📌 embarked: заполнено 'unknown'")

print("\n✅ Пропуски заполнены константами.")
print("✅ Позволяет модели явно видеть, где были пропуски.")

In [ ]:
# ЯЧЕЙКА 9: СРАВНЕНИЕ МЕТОДОВ ЗАПОЛНЕНИЯ (ВИЗУАЛИЗАЦИЯ)

print("=" * 60)
print("📊 СРАВНЕНИЕ МЕТОДОВ ЗАПОЛНЕНИЯ (на примере Age)")
print("=" * 60)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

# Оригинальные данные (с пропусками)
axes[0].hist(df['age'].dropna(), bins=30, color='gray', alpha=0.7, edgecolor='black')
axes[0].set_title('Оригинал (с пропусками)')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Частота')

# Mean Imputation
axes[1].hist(df_mean_imp['age'], bins=30, color='#4ECDC4', alpha=0.7, edgecolor='black')
axes[1].axvline(df_mean_imp['age'].mean(), color='red', linestyle='--', linewidth=2, label=f'mean={df_mean_imp["age"].mean():.1f}')
axes[1].set_title('Mean Imputation')
axes[1].set_xlabel('Age')
axes[1].legend()

# Median Imputation
axes[2].hist(df_median_imp['age'], bins=30, color='#45B7D1', alpha=0.7, edgecolor='black')
axes[2].axvline(df_median_imp['age'].median(), color='red', linestyle='--', linewidth=2, label=f'median={df_median_imp["age"].median():.1f}')
axes[2].set_title('Median Imputation')
axes[2].set_xlabel('Age')
axes[2].legend()

# Constant Imputation (-1)
axes[3].hist(df_const_imp['age'], bins=30, color='#96CEB4', alpha=0.7, edgecolor='black')
axes[3].set_title('Constant Imputation (-1)')
axes[3].set_xlabel('Age')

# Сравнение статистик
stats_data = {
    'Метод': ['Оригинал', 'Mean', 'Median', 'Constant'],
    'Среднее': [df['age'].mean(), df_mean_imp['age'].mean(), df_median_imp['age'].mean(), df_const_imp['age'].mean()],
    'Медиана': [df['age'].median(), df_mean_imp['age'].median(), df_median_imp['age'].median(), df_const_imp['age'].median()],
    'Ст.откл.': [df['age'].std(), df_mean_imp['age'].std(), df_median_imp['age'].std(), df_const_imp['age'].std()]
}
stats_df = pd.DataFrame(stats_data).round(2)

# Скрываем 4-й график для таблицы
axes[4].axis('off')
axes[5].axis('off')

# Таблица со статистиками на месте 5-го графика
table = axes[5].table(cellText=stats_df.values, colLabels=stats_df.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
axes[5].axis('off')
axes[5].set_title('Сравнение статистик', fontsize=14)

plt.suptitle('Сравнение методов заполнения пропусков (Age)', fontsize=16)
plt.tight_layout()
plt.show()

### 📌 Часть 2: Кодирование категориальных признаков

In [ ]:
# ЯЧЕЙКА 10: ПОДГОТОВКА ДАННЫХ ДЛЯ КОДИРОВАНИЯ

# Берём данные после заполнения пропусков медианой (для числовых) и модой (для категориальных)
df_clean = df.copy()

# Заполняем числовые пропуски медианой
for col in ['age', 'fare']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Заполняем категориальные пропуски модой
df_clean['embarked'] = df_clean['embarked'].fillna(df_clean['embarked'].mode()[0])

print("✅ Данные очищены от пропусков.")

# Категориальные признаки для кодирования
categorical_cols = ['sex', 'embarked', 'pclass']

print("\n📋 Категориальные признаки:")
for col in categorical_cols:
    print(f"  {col}: {df_clean[col].unique().tolist()}")

In [ ]:
# ЯЧЕЙКА 11: МЕТОД 1 — LABEL ENCODING (Порядковое кодирование)

print("=" * 60)
print("🔹 МЕТОД 1: LABEL ENCODING (порядковое кодирование)")
print("=" * 60)

df_label = df_clean.copy()
le = LabelEncoder()

for col in categorical_cols:
    df_label[col + '_label'] = le.fit_transform(df_label[col])
    print(f"📌 {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print("\n✅ Категориальные признаки закодированы числами (0, 1, 2, ...).")
print("⚠️ Недостаток: алгоритм может воспринять порядок как смысловой (например, 2 > 1).")

print("\n📋 Результат:")
display(df_label[[col + '_label' for col in categorical_cols]].head(10))

In [ ]:
# ЯЧЕЙКА 12: МЕТОД 2 — ONE-HOT ENCODING (Бинарное кодирование)

print("=" * 60)
print("🔹 МЕТОД 2: ONE-HOT ENCODING (бинарное кодирование)")
print("=" * 60)

from sklearn.preprocessing import OneHotEncoder

df_ohe = df_clean.copy()

# Применяем One-Hot Encoding
ohe = OneHotEncoder(sparse_output=False, dtype=int)
ohe_result = ohe.fit_transform(df_ohe[categorical_cols])

# Создаём DataFrame с результатами
ohe_columns = []
for i, col in enumerate(categorical_cols):
    for category in ohe.categories_[i]:
        ohe_columns.append(f"{col}_{category}")

df_ohe_encoded = pd.DataFrame(ohe_result, columns=ohe_columns, index=df_ohe.index)

# Объединяем с исходными данными
df_ohe_final = pd.concat([df_ohe, df_ohe_encoded], axis=1)

print(f"📌 Было {len(categorical_cols)} категориальных признаков")
print(f"📌 Стало {len(ohe_columns)} бинарных признаков")
print(f"📌 Категории: {ohe.categories_}")

print("\n📋 Результат (первые 5 строк):")
display(df_ohe_final[categorical_cols + ohe_columns].head())

print("\n✅ Категориальные признаки преобразованы в набор бинарных столбцов.")
print("✅ Нет ложного порядка — корректно для любых алгоритмов.")
print("⚠️ Недостаток: при большом количестве категорий резко растёт число признаков.")

In [ ]:
# ЯЧЕЙКА 13: СРАВНЕНИЕ МЕТОДОВ КОДИРОВАНИЯ

print("=" * 60)
print("📊 СРАВНЕНИЕ МЕТОДОВ КОДИРОВАНИЯ")
print("=" * 60)

comparison_data = {
    'Метод': ['Label Encoding', 'One-Hot Encoding'],
    'Исходных признаков': [len(categorical_cols), len(categorical_cols)],
    'Итоговых признаков': [len(categorical_cols), len(ohe_columns)],
    'Сохраняет порядок?': ['Да (проблематично)', 'Нет'],
    'Подходит для tree-based': ['Да', 'Да'],
    'Подходит для линейных': ['Нет', 'Да']
}
comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

print("\n📌 Рекомендации:")
print("  • Label Encoding — для порядковых категорий (например, small/medium/large)")
print("  • One-Hot Encoding — для номинальных категорий (например, пол, город)")

### 📌 Часть 3: Масштабирование данных

In [ ]:
# ЯЧЕЙКА 14: ПОДГОТОВКА ДАННЫХ ДЛЯ МАСШТАБИРОВАНИЯ

# Выбираем числовые признаки для масштабирования
numeric_features = ['age', 'fare', 'sibsp', 'parch']

# Создаём копию с заполненными пропусками
df_scaled = df_clean.copy()

# Проверяем, что пропусков нет
print("Проверка пропусков в числовых признаках:")
for col in numeric_features:
    print(f"  {col}: {df_scaled[col].isnull().sum()} пропусков")

print("\n📊 Исходные данные (первые 5 строк):")
display(df_scaled[numeric_features].head())

In [ ]:
# ЯЧЕЙКА 15: МЕТОД 1 — STANDARDIZATION (Стандартизация)

print("=" * 60)
print("🔹 МЕТОД 1: STANDARDIZATION (Z-score нормализация)")
print("=" * 60)

scaler_standard = StandardScaler()
df_standard = df_scaled.copy()
df_standard[numeric_features] = scaler_standard.fit_transform(df_scaled[numeric_features])

print("📊 Статистика после стандартизации:")
for col in numeric_features:
    print(f"  {col}: mean={df_standard[col].mean():.4f}, std={df_standard[col].std():.4f}")

print("\n📋 Результат (первые 5 строк):")
display(df_standard[numeric_features].head())

print("\n✅ Данные приведены к среднему 0 и стандартному отклонению 1.")
print("✅ Подходит для линейных моделей, SVM, PCA.")

In [ ]:
# ЯЧЕЙКА 16: МЕТОД 2 — MIN-MAX SCALING (Нормализация)

print("=" * 60)
print("🔹 МЕТОД 2: MIN-MAX SCALING (нормализация в диапазон [0, 1])")
print("=" * 60)

scaler_minmax = MinMaxScaler()
df_minmax = df_scaled.copy()
df_minmax[numeric_features] = scaler_minmax.fit_transform(df_scaled[numeric_features])

print("📊 Статистика после Min-Max Scaling:")
for col in numeric_features:
    print(f"  {col}: min={df_minmax[col].min():.4f}, max={df_minmax[col].max():.4f}")

print("\n📋 Результат (первые 5 строк):")
display(df_minmax[numeric_features].head())

print("\n✅ Данные приведены к диапазону [0, 1].")
print("✅ Подходит для нейронных сетей, методов, чувствительных к масштабу.")

In [ ]:
# ЯЧЕЙКА 17: МЕТОД 3 — ROBUST SCALING (Устойчивое масштабирование)

print("=" * 60)
print("🔹 МЕТОД 3: ROBUST SCALING (устойчивое к выбросам)")
print("=" * 60)

scaler_robust = RobustScaler()
df_robust = df_scaled.copy()
df_robust[numeric_features] = scaler_robust.fit_transform(df_scaled[numeric_features])

print("📊 Статистика после Robust Scaling:")
for col in numeric_features:
    print(f"  {col}: median={df_robust[col].median():.4f}, IQR={df_robust[col].quantile(0.75) - df_robust[col].quantile(0.25):.4f}")

print("\n📋 Результат (первые 5 строк):")
display(df_robust[numeric_features].head())

print("\n✅ Использует медиану и IQR, устойчив к выбросам.")
print("✅ Лучший выбор, если в данных есть аномалии.")

In [ ]:
# ЯЧЕЙКА 18: ВИЗУАЛИЗАЦИЯ МЕТОДОВ МАСШТАБИРОВАНИЯ

print("=" * 60)
print("📊 СРАВНЕНИЕ МЕТОДОВ МАСШТАБИРОВАНИЯ (на примере Age и Fare)")
print("=" * 60)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Оригинальные данные
axes[0, 0].scatter(df_scaled['age'], df_scaled['fare'], alpha=0.5, s=30)
axes[0, 0].set_title('Оригинал')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Fare')

# Standardization
axes[0, 1].scatter(df_standard['age'], df_standard['fare'], alpha=0.5, s=30, color='#4ECDC4')
axes[0, 1].set_title('Standardization')
axes[0, 1].set_xlabel('Age (scaled)')
axes[0, 1].set_ylabel('Fare (scaled)')

# Min-Max Scaling
axes[0, 2].scatter(df_minmax['age'], df_minmax['fare'], alpha=0.5, s=30, color='#45B7D1')
axes[0, 2].set_title('Min-Max Scaling')
axes[0, 2].set_xlabel('Age (scaled)')
axes[0, 2].set_ylabel('Fare (scaled)')

# Robust Scaling
axes[0, 3].scatter(df_robust['age'], df_robust['fare'], alpha=0.5, s=30, color='#96CEB4')
axes[0, 3].set_title('Robust Scaling')
axes[0, 3].set_xlabel('Age (scaled)')
axes[0, 3].set_ylabel('Fare (scaled)')

# Гистограммы для Age
axes[1, 0].hist(df_scaled['age'], bins=30, color='gray', alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Age (оригинал)')
axes[1, 0].set_xlabel('Age')

axes[1, 1].hist(df_standard['age'], bins=30, color='#4ECDC4', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Age (Standardization)')
axes[1, 1].set_xlabel('Age (scaled)')

axes[1, 2].hist(df_minmax['age'], bins=30, color='#45B7D1', alpha=0.7, edgecolor='black')
axes[1, 2].set_title('Age (Min-Max)')
axes[1, 2].set_xlabel('Age (scaled)')

axes[1, 3].hist(df_robust['age'], bins=30, color='#96CEB4', alpha=0.7, edgecolor='black')
axes[1, 3].set_title('Age (Robust)')
axes[1, 3].set_xlabel('Age (scaled)')

plt.suptitle('Сравнение методов масштабирования', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# ЯЧЕЙКА 19: СРАВНИТЕЛЬНАЯ ТАБЛИЦА МЕТОДОВ МАСШТАБИРОВАНИЯ

print("=" * 60)
print("📊 СРАВНИТЕЛЬНАЯ ТАБЛИЦА МЕТОДОВ МАСШТАБИРОВАНИЯ")
print("=" * 60)

scaling_comparison = {
    'Метод': ['StandardScaler', 'MinMaxScaler', 'RobustScaler'],
    'Формула': ['(x - μ) / σ', '(x - min) / (max - min)', '(x - Q2) / (Q3 - Q1)'],
    'Диапазон': ['~[-3, 3]', '[0, 1]', '~[-1.5, 1.5]'],
    'Чувствительность к выбросам': ['Высокая', 'Высокая', 'Низкая'],
    'Когда использовать': ['Линейные модели, SVM', 'Нейросети, методы с градиентом', 'Данные с выбросами']
}
display(pd.DataFrame(scaling_comparison))

### 📌 Часть 4: Полный пайплайн предобработки

In [ ]:
# ЯЧЕЙКА 20: ПОЛНЫЙ ПАЙПЛАЙН ПРЕДОБРАБОТКИ

print("=" * 60)
print("🔧 ПОЛНЫЙ ПАЙПЛАЙН ПРЕДОБРАБОТКИ (с использованием ColumnTransformer)")
print("=" * 60)

# Определяем числовые и категориальные признаки
numeric_features = ['age', 'fare', 'sibsp', 'parch']
categorical_features = ['pclass', 'sex', 'embarked']

# Пайплайн для числовых признаков: заполнение медианой + стандартизация
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Пайплайн для категориальных признаков: заполнение модой + One-Hot
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Объединяем в ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Применяем к данным
X_processed = preprocessor.fit_transform(df_clean)

# Получаем имена признаков после обработки
feature_names = (
    numeric_features +
    list(preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features))
)

df_processed = pd.DataFrame(X_processed, columns=feature_names)

print(f"📌 Исходных признаков: {len(df_clean.columns)}")
print(f"📌 После обработки: {len(df_processed.columns)}")
print(f"\n📋 Итоговые признаки:")
print(df_processed.columns.tolist())

print("\n📋 Результат (первые 5 строк):")
display(df_processed.head())

In [ ]:
# ЯЧЕЙКА 21: ИТОГОВАЯ СВОДКА

print("=" * 60)
print("📊 ИТОГОВАЯ СВОДКА ВЫПОЛНЕННЫХ ЗАДАЧ")
print("=" * 60)

summary = {
    'Задача': [
        '1. Обработка пропусков',
        '2. Кодирование категориальных признаков',
        '3. Масштабирование данных',
        '4. Полный пайплайн'
    ],
    'Рассмотренные методы': [
        'Удаление строк, Mean, Median, Mode, Constant',
        'Label Encoding, One-Hot Encoding',
        'StandardScaler, MinMaxScaler, RobustScaler',
        'ColumnTransformer + Pipeline'
    ],
    'Итоговый признак': [
        'Нет пропусков',
        'Все признаки числовые',
        'Все признаки в одном масштабе',
        'Готово для обучения модели'
    ]
}
display(pd.DataFrame(summary))

print("\n✅ Лабораторная работа №2 успешно выполнена!")